# Enrollment API — demo

In [ ]:
import httpx

API = "http://localhost:8000/api/v1"
API_V2 = "http://localhost:8000/api/v2"

## 1. Read endpoints (200 OK)

In [ ]:
# Get one student
r = httpx.get(f"{API}/students/1")
r.status_code, r.json()

In [ ]:
# Browse offerings that are open for registration (paginated)
r = httpx.get(f"{API}/offerings", params={"open_only": True, "page": 1, "size": 3})
r.status_code, r.json()

## 2. Register a course (201 Created) + Idempotency-Key

Student 1 registers for offering 101 (in the current open term). The
`Idempotency-Key` makes the request safe to retry.

In [ ]:
r = httpx.post(
    f"{API}/enrollments",
    json={"student_id": 1, "offering_id": 101},
    headers={"Idempotency-Key": "demo-key-1"},
)
r.status_code, r.json()

## 3. Idempotency — same key returns the same enrollment

The same request with the same key replays the original result instead of
creating a duplicate (same `id`).

In [ ]:
r = httpx.post(
    f"{API}/enrollments",
    json={"student_id": 1, "offering_id": 101},
    headers={"Idempotency-Key": "demo-key-1"},
)
r.status_code, r.json()

## 4. Error handling — 404 (RFC 7807)

A resource that does not exist returns 404 as `application/problem+json`.

In [ ]:
r = httpx.get(f"{API}/students/999999")
r.status_code, r.json()

## 5. Error handling — 422 (RFC 7807)

A body that fails validation (`offering_id` missing) returns 422, also
`problem+json`, with an `errors` array.

In [ ]:
r = httpx.post(f"{API}/enrollments", json={"student_id": 1})
r.status_code, r.json()

## 6. API versioning — v1 vs v2

Same student, two contracts mounted side by side: v1 uses
`student_code` / `full_name` / timestamps; v2 renames them to `code` / `name`
and drops the timestamps.

In [ ]:
# v1 contract
httpx.get(f"{API}/students/1").json()

In [ ]:
# v2 contract
httpx.get(f"{API_V2}/students/1").json()